# Explorer 03 - Formats and reissues

**Purpose.** Explore the format mix by year and settle on a definition of reissue.

**Feeds main report section:** 5.2 RQ2 - Format mix and the vinyl boom

**Modules used:** `vinyl.formats`, `vinyl.viz`

**Tables read:** `release_formats`, `releases`, `masters`, `release_genres`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-03-formats")
from vinyl import formats, viz

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_formats = ingest.read_table(spark, snapshot, "release_formats")
releases = ingest.read_table(spark, snapshot, "releases")
masters = ingest.read_table(spark, snapshot, "masters")
release_genres = ingest.read_table(spark, snapshot, "release_genres")

## 1. What formats and descriptions exist?

In [ ]:
release_formats.groupBy("format_name").agg(F.countDistinct("release_id").alias("n")).orderBy(F.desc("n")).show(30, truncate=False)

In [ ]:
release_formats.select(F.explode("descriptions").alias("description")).groupBy("description").count().orderBy(F.desc("count")).show(40, truncate=False)

## 2. Format shares by year

In [ ]:
format_shares = formats.format_shares_by_year(release_formats).toPandas()
main_formats = formats.main_formats(format_shares, min_share=0.01)
print("Main formats (>= 1% of releases in at least one full year, largest mean share first):")
print(main_formats)

In [ ]:
grouped_shares = formats.format_shares_by_year(
    formats.group_minor_formats(release_formats, main_formats)
).toPandas()
ax = viz.plot_lines(
    grouped_shares, "year", "share", "format_name",
    title="Share of releases by format, 2016-2026 (minor formats grouped as Other; 2026 partial)",
    ylabel="share of the year's releases", percent=True,
)
ax.axvspan(config.LAST_FULL_YEAR + 0.5, config.YEAR_MAX + 0.5, color="grey", alpha=0.15)
ax.text(config.YEAR_MAX + 0.4, ax.get_ylim()[1], "2026 partial\n(Jan-Sep)", ha="right", va="top", fontsize=8)

## 3. Reissue definitions

Compare the tag-based flag (`is_reissue`) with the master-based flag (`is_reissue_master`) before picking one.

In [ ]:
flagged = formats.add_master_reissue_flag(release_formats, releases, masters, min_gap_years=1)
flagged.groupBy("is_reissue", "is_reissue_master").agg(F.countDistinct("release_id").alias("n")).show()

In [ ]:
vinyl = flagged.where(F.col("format_name") == "Vinyl")
print(f"End year for comparisons: {config.LAST_FULL_YEAR} (2026 is partial)")
for flag in ["is_reissue", "is_reissue_master", "is_reissue_any"]:
    print(flag)
    formats.reissue_share(vinyl, by=["year"], flag_col=flag).orderBy("year").show()

### Share of vinyl growth explained by reissues

`(reissues_end - reissues_start) / (releases_end - releases_start)` for vinyl, from `config.YEAR_MIN` to `config.LAST_FULL_YEAR`, for each flag. Null means vinyl releases did not grow.

In [ ]:
from functools import reduce

by_flag = [
    formats.reissue_share(vinyl, by=["year"], flag_col=flag).withColumn("flag", F.lit(flag))
    for flag in ["is_reissue", "is_reissue_master", "is_reissue_any"]
]
growth = formats.growth_explained_by_reissues(
    reduce(lambda a, b: a.unionByName(b), by_flag), config.YEAR_MIN, config.LAST_FULL_YEAR
)
growth.orderBy("flag").show(truncate=False)

## 4. Reissue share by genre

In [ ]:
by_genre = formats.reissue_share(
    flagged.join(release_genres.select("release_id", "genre"), "release_id"),
    by=["year", "format_name", "genre"],
    flag_col="is_reissue_any",
)
by_genre.where(F.col("format_name") == "Vinyl").orderBy("genre", "year").show(40)

## 5. How old is the music being reissued?

Age is `year - master_year`, for releases that have a master. A master dated after the release is left out and counted below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

age = {fmt: formats.reissue_age_distribution(flagged, fmt).toPandas() for fmt in ["Vinyl", "CD"]}
print("Distinct releases with master_year later than the release year (left out):",
      {fmt: formats.count_master_after_release(flagged, fmt) for fmt in age})

labels = formats.age_bin_labels()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (fmt, pdf) in zip(axes, age.items()):
    pdf = pdf.assign(age_bin=pd.Categorical(pdf["age_bin"], categories=labels, ordered=True))
    viz.plot_stacked_share(
        pdf, "year", "n_releases", "age_bin",
        title=f"{fmt}: age of the master in years (releases with a master; 2026 partial)", ax=ax,
    )
fig.tight_layout()
pd.DataFrame({fmt: pdf.groupby("year")["median_age"].first() for fmt, pdf in age.items()})

## 6. Pressing plants

Check the spelling of the role in `entity_type_name` before using it.

In [ ]:
release_companies = ingest.read_table(spark, snapshot, "release_companies")
(release_companies.where(F.col("entity_type_name").rlike("(?i)press|manufactur"))
    .groupBy("entity_type", "entity_type_name").agg(F.countDistinct("release_id").alias("n"))
    .orderBy(F.desc("n")).show(truncate=False))
role = "Pressed By"

In [ ]:
plants = formats.top_pressing_plants(release_companies, release_formats, "Vinyl", role, top_n=15).toPandas()
display(plants.drop_duplicates("company_id").sort_values("rank")[["rank", "company_id", "company_name", "total_full_years"]])
ax = viz.plot_lines(
    plants, "year", "share_of_format_year", "company_name",
    title=f"Top 15 '{role}' companies: share of vinyl releases that name a '{role}' company (2026 partial)",
    percent=True,
)
ax.axvspan(config.LAST_FULL_YEAR + 0.5, config.YEAR_MAX + 0.5, color="grey", alpha=0.15)

In [ ]:
# Coverage: share of vinyl releases per year that name any pressing plant at all.
formats.pressing_plant_coverage(release_companies, release_formats, "Vinyl", role).orderBy("year").show(truncate=False)

## 7. Digital ("File") releases

In [ ]:
formats.format_coverage(release_formats, releases, "File").orderBy("year").show(truncate=False)

**Reading rule.** If the File share jumps or collapses in step with catalogue policy rather than smoothly, File is not comparable across years. Read `share`, `desc_coverage` and `top_descriptions` together for such breaks.

## Decisions to carry into `main.ipynb`

Filled in from the full-data runs in `executed/`. Each line should end up as a sentence in the report's methodology or limitations.

- [x] Reissue definition: `is_reissue_any` (a Reissue, Repress or Remastered tag, or a master at least 1 year older than the release), with the tag-only and master-only definitions reported next to it. The two flags miss different releases: 214,206 releases are reissues by master only and 28,763 by tag only (section 3).
- [x] Formats shown: File, Vinyl, CD, Cassette, CDr, All Media and DVD, the formats with at least 1% of releases in at least one full year (`formats.main_formats(min_share=0.01)`); all others are grouped as Other (1.7-2.2% of releases per year).
- [x] Share of vinyl growth explained by reissues, 2016 to 2025: 45.1% with `is_reissue_any` (vinyl releases 73,730 to 92,779, reissues 21,353 to 29,936); 27.4% with tags only and 42.7% with masters only.